In [1]:
import torch
from transformers import AutoTokenizer

from eshmun.models.zero.models import EshmunZeroForMaskedLM

In [2]:
device = torch.device("cpu" if torch.cuda.is_available() else 'cpu')

device

device(type='cpu')

In [4]:
!ls /tmp/eshmun-zero-mlm/

checkpoint-6500  checkpoint-7239  model.safetensors	 training_args.bin
checkpoint-7000  config.json	  tokenizer_config.json  vocab.txt


In [5]:
model_path = '/tmp/eshmun-zero-mlm/'
model_path = '/tmp/eshmun-zero-mlm/checkpoint-7239'

In [7]:
model = EshmunZeroForMaskedLM.from_pretrained(model_path).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

You are using a model of type `eshmun_zero` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


In [8]:
sample_sequence = 'MDNEIIIIVIVIIIFFFYLKQKKLTNCETQVVKVQKDIDEINLKLKKLNK'
masked_sequence = 'MDNEIII<mask>VIVIIIFFFYL<mask>QKKLTNCETQVVKVQK<mask>IDE<mask>NL<mask>LKKLNK'

In [15]:
inputs = tokenizer(masked_sequence, return_tensors='pt').to(device)

labels = tokenizer(sample_sequence, return_tensors='pt')['input_ids'].to(device)

In [16]:
output = model(
    input_ids=inputs['input_ids'],
    attention_mask=inputs['attention_mask'],
    labels=labels
)

In [17]:
output.loss.exp()

tensor(2.5847, grad_fn=<ExpBackward0>)

In [24]:
inputs = tokenizer(sample_sequence, return_tensors="pt").to(device)

# 2. Create masked input — mask a few positions manually
input_ids = inputs["input_ids"].clone()
labels = input_ids.clone()

mask_positions = [8, 20, 37, 41, 44]
mask_token_id = tokenizer.mask_token_id  # usually <mask>

for pos in mask_positions:
    input_ids[0, pos] = mask_token_id

# 3. Forward pass — NO labels needed for inference
with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=inputs["attention_mask"]
    )
    logits = outputs.logits  # shape: [1, seq_len, vocab_size]

# 4. For each masked position, show top 5 predictions
print(f"{'Position':<12} {'Rank':<6} {'Token':<10} {'Prob':>8}")
print("-" * 40)

for pos in mask_positions:
    token_logits = logits[0, pos]                          # [vocab_size]
    probs = torch.softmax(token_logits, dim=-1)                # [vocab_size]
    top5 = torch.topk(probs, k=3)

    true_token = tokenizer.convert_ids_to_tokens([labels[0, pos].item()])[0]
    print(f"\nPosition {pos} (true: {true_token})")

    for rank, (prob, idx) in enumerate(zip(top5.values, top5.indices)):
        token = tokenizer.convert_ids_to_tokens([idx.item()])[0]
        bar = "█" * int(prob.item() * 40)
        print(f"  {rank+1}. {token:<8} {prob.item():.4f}  {bar}")


Position     Rank   Token          Prob
----------------------------------------

Position 8 (true: I)
  1. I        0.2788  ███████████
  2. L        0.1916  ███████
  3. F        0.1420  █████

Position 20 (true: K)
  1. I        0.1787  ███████
  2. L        0.1576  ██████
  3. F        0.1167  ████

Position 37 (true: D)
  1. K        0.1410  █████
  2. N        0.1141  ████
  3. E        0.1123  ████

Position 41 (true: I)
  1. K        0.1497  █████
  2. N        0.1076  ████
  3. E        0.1049  ████

Position 44 (true: K)
  1. K        0.1738  ██████
  2. N        0.1106  ████
  3. E        0.1078  ████
